# 01. Data Inspection

## Purpose
Before any cleaning or analysis, this notebook takes a first honest look at both raw datasets: Cohort 9 and Cohort 10 registration exports. The goal here is not to fix anything yet, only to observe. What columns exist, what data types pandas assigned them, where values are missing, and where something looks structurally off.

Decisions about what to clean and how come after this inspection, not before it.

## Setup

In [5]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## Load Raw Data

Loading both cohort files directly from `data/raw/`, untouched from what was provided in the challenge.

In [6]:
cohort9 = pd.read_csv('../data/raw/cohort_9_raw.csv')
cohort10 = pd.read_csv('../data/raw/cohort_10_raw.csv')

print("Cohort 9 shape:", cohort9.shape)
print("Cohort 10 shape:", cohort10.shape)

Cohort 9 shape: (1147, 9)
Cohort 10 shape: (171, 9)


## Cohort 9: Structure

Checking column names, data types, and non-null counts. Cohort 9's form did not collect Occupation, so that column is expected to be fully missing here.

In [7]:
cohort9.info()
cohort9.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1147 entries, 0 to 1146
Data columns (total 9 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Timestamp                                 1147 non-null   object 
 1   NAME                                      1092 non-null   object 
 2   Email address                             1147 non-null   object 
 3   Phone Number (Add country code e.g +234)  1146 non-null   object 
 4   Country (e.g. Nigeria, UK)                1147 non-null   object 
 5   Gender                                    1147 non-null   object 
 6   Choice of Program                         1147 non-null   object 
 7   How did you hear about our Program?       1147 non-null   object 
 8   Amount Paid                               0 non-null      float64
dtypes: float64(1), object(8)
memory usage: 80.8+ KB


,Timestamp,NAME,Email address,Phone Number (Add country code e.g +234),"Country (e.g. Nigeria, UK)",Gender,Choice of Program,How did you hear about our Program?,Amount Paid
0,11/6/25 10:49 AM,Zakariyau,muky@gmail.com,+23480,Nigeria,Male,Data Science and AI,X (Twitter),NaN
1,11/7/25 9:16 PM,Janice,jani@gmail.com,+23490,Nigeria,Female,Healthcare Data Analytics,Referral,NaN
2,11/10/25 9:45 AM,Innocent,inno@gmail.com,27839,South Africa,Male,Data Science and AI,X (Twitter),NaN
3,11/10/25 10:00 AM,Keolebogile,keol@gmail.com,+26772,Botswana Africa,Male,Data Science and AI,LinkedIn,NaN
4,11/10/25 10:11 AM,Noxolo,feli@gmail.com,+36204,South Africa,Female,Data Science and AI,LinkedIn,NaN


## Cohort 10: Structure

Same check for Cohort 10. This cohort's form did collect Occupation directly, and does not include an Amount Paid column.

In [9]:
cohort10.info()
cohort10.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171 entries, 0 to 170
Data columns (total 9 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   Timestamp                                  171 non-null    object
 1   NAME                                       171 non-null    object
 2   Gender                                     171 non-null    object
 3   EMAIL ADDRESS                              171 non-null    object
 4   PHONE NUMBER  (Add country code e.g +234)  170 non-null    object
 5   COUNTRY  (e.g. Nigeria, UK)                171 non-null    object
 6   CHOICE OF COURSE                           171 non-null    object
 7   Occupation                                 170 non-null    object
 8   How did you hear about our Program?        171 non-null    object
dtypes: object(9)
memory usage: 12.2+ KB


,Timestamp,NAME,Gender,EMAIL ADDRESS,PHONE NUMBER (Add country code e.g +234),"COUNTRY (e.g. Nigeria, UK)",CHOICE OF COURSE,Occupation,How did you hear about our Program?
0,5/6/2026 11:03:32,Sanni,Male,sann@gmail.com,+23481,Nigeria,Financial Analytics,Data analyst,Linkedin
1,5/6/2026 15:19:31,lydia,Female,imal@gmail.com,+25677,Uganda,Data science and ML,IT,Linkedin
2,5/6/2026 17:02:15,Ifedolapo,Female,Ifed@gmail.com,+23481,Nigeria,Healthcare Data Analytics,Student,X (Twitter)
3,5/6/2026 22:54:00,Okhakumhe,Female,ochu@gmail.com,+23470,Nigeria,Supply Chain Analytics,Pharmacist,Linkedin
4,5/7/2026 12:14:57,Valarie,Female,vala@gmail.com,+25479,Kenya,Data science and ML,Data Analyst,X (Twitter)


## Initial Findings

- Cohort 9: 1147 rows, but only 1092 have a Name. Other fields (Email, Phone, Country) are mostly present even where Name is missing — investigating below.
- Cohort 9: Amount Paid column is entirely empty (0 non-null). Not usable for analysis.
- Both cohorts: Timestamp is stored as text, not datetime. Needs conversion.
- Column names differ in capitalization and wording between cohorts (e.g. "Choice of Program" vs "CHOICE OF COURSE"). Will need to be standardized before combining or comparing cohorts.

## Investigating Missing Names (Cohort 9)

In [10]:
cohort9[cohort9['NAME'].isna()].head(10)

,Timestamp,NAME,Email address,Phone Number (Add country code e.g +234),"Country (e.g. Nigeria, UK)",Gender,Choice of Program,How did you hear about our Program?,Amount Paid
9,11/10/25 12:16 PM,NaN,saul@gmail.com,+27685,South Africa,Female,Financial Analytics,LinkedIn,NaN
24,11/10/25 1:52 PM,NaN,aden@gmail.com,234814,UK,Male,Data Science and AI,LinkedIn,NaN
48,11/10/25 6:14 PM,NaN,coli@gmail.com,(+27)684,Sout,Male,Financial Analytics,X (Twitter),NaN
50,11/10/25 6:32 PM,NaN,ivan@gmail.com,+25471,Kenya,Male,Healthcare Data Analytics,X (Twitter),NaN
243,11/13/25 11:28 PM,NaN,abdu@gmail.com,090583,Nigeria,Male,Financial Analytics,WhatsApp,NaN
308,11/18/25 12:50 AM,NaN,gift@gmail.com,+234 9,Nigeria,Female,Sales and Marketing Analytics,X (Twitter),NaN
345,11/24/25 11:27 AM,NaN,ogun@gmail.com,070118,Nigeria,Male,Financial Analytics,X (Twitter),NaN
346,11/24/25 12:31 PM,NaN,fait@gmail.com,+27723,South Africa,Female,Financial Analytics,X (Twitter),NaN
354,11/24/25 3:28 PM,NaN,sech@gmail.com,+86132,China,Male,Data Science and AI,X (Twitter),NaN
393,11/25/25 11:15 AM,NaN,Owir@gmail.com,050774,Ghana,Female,Data Science and AI,WhatsApp,NaN


## Pattern Check: Email Usernames on Missing-Name Rows

The missing-name rows all show short, generic email usernames (4-5 letters, lowercase). Checking whether this is consistent across all missing-name rows, and how many rows are affected in total.

In [11]:
missing_names = cohort9[cohort9['NAME'].isna()].copy()
print("Total missing-name rows:", len(missing_names))

missing_names['email_username'] = missing_names['Email address'].str.split('@').str[0]
missing_names['username_length'] = missing_names['email_username'].str.len()

missing_names['username_length'].describe()

Total missing-name rows: 55


count    55.0
mean      4.0
std       0.0
min       4.0
25%       4.0
50%       4.0
75%       4.0
max       4.0
Name: username_length, dtype: float64

In [12]:
missing_names[['Email address', 'email_username', 'username_length']].sort_values('username_length')

,Email address,email_username,username_length
9,saul@gmail.com,saul,4
24,aden@gmail.com,aden,4
48,coli@gmail.com,coli,4
50,ivan@gmail.com,ivan,4
243,abdu@gmail.com,abdu,4
308,gift@gmail.com,gift,4
345,ogun@gmail.com,ogun,4
346,fait@gmail.com,fait,4
354,sech@gmail.com,sech,4
393,Owir@gmail.com,Owir,4


## Finding: Deliberate De-identification, Not Missing Data

All 55 rows with a missing Name also have an email username of exactly 4 characters (std = 0 across the group), compared to normal, varied-length email usernames elsewhere in the dataset. Every other field on these rows (Phone, Country, Gender, Course, Channel) is fully present.

This is not random missingness. It matches the same kind of de-identification the challenge brief described for phone numbers, applied here to Name and email without being explicitly documented.

**Decision:** These rows are kept in the dataset. They contain valid, usable data for every field relevant to this analysis (channel, course, country, timing). Name is not used in any part of this analysis, so its absence has no impact on the findings. This will be noted as a documented assumption rather than treated as a data quality problem to fix.

In [13]:
missing_names['Timestamp'].sort_values()

865      1/22/26 10:03 PM
883      1/22/26 10:47 PM
889      1/22/26 10:51 PM
899      1/22/26 11:11 PM
902      1/22/26 11:21 PM
906      1/22/26 11:28 PM
649       1/22/26 4:36 PM
658       1/22/26 5:22 PM
663       1/22/26 5:40 PM
685       1/22/26 6:56 PM
687       1/22/26 6:57 PM
702       1/22/26 7:09 PM
711       1/22/26 7:14 PM
730       1/22/26 7:40 PM
745       1/22/26 7:57 PM
754       1/22/26 8:04 PM
758       1/22/26 8:07 PM
767       1/22/26 8:15 PM
785       1/22/26 8:28 PM
805       1/22/26 8:53 PM
824       1/22/26 9:09 PM
835       1/22/26 9:18 PM
846       1/22/26 9:29 PM
847       1/22/26 9:29 PM
849       1/22/26 9:29 PM
854       1/22/26 9:39 PM
856       1/22/26 9:44 PM
1045     1/23/26 10:28 AM
1046     1/23/26 10:29 AM
942       1/23/26 1:18 AM
950       1/23/26 1:57 AM
987       1/23/26 7:23 AM
1010      1/23/26 9:07 AM
1016      1/23/26 9:28 AM
9       11/10/25 12:16 PM
24       11/10/25 1:52 PM
48       11/10/25 6:14 PM
50       11/10/25 6:32 PM
243     11/1

## Finding: Two Distinct Patterns Within the Missing-Name Rows

Sorting the 55 missing-name rows by timestamp reveals two separate groups:

- Approximately 34 rows cluster tightly within 22-23 Jan 2026, several submitted only minutes apart. This pattern is inconsistent with individual, organic form submissions and more likely reflects a bulk import or batch registration event.
- The remaining approximately 21 rows are spread thinly from 10 Nov 2025 through 30 Dec 2025, roughly one every few days, consistent with the normal registration pace seen elsewhere in the cohort.

Both groups still carry usable data in every field this analysis relies on (Country, Course, Channel, Gender). No rows will be dropped on this basis alone, but the Jan 22-23 cluster is worth flagging separately in the timing/trend analysis, since a batch import could distort any "registrations over time" pattern if not accounted for.

In [14]:
jan_cluster = missing_names[missing_names['Timestamp'].str.startswith(('1/22/26', '1/23/26'))]
print("Rows in the Jan 22-23 cluster:", len(jan_cluster))
jan_cluster['How did you hear about our Program?'].value_counts()

Rows in the Jan 22-23 cluster: 34


How did you hear about our Program?
X (Twitter)    29
Referral        2
WhatsApp        2
LinkedIn        1
Name: count, dtype: int64

## Finding: X (Twitter) Spike Tied to the Jan 22-23 Cluster

29 of the 34 rows in the Jan 22-23 cluster came through X (Twitter) — the same channel, the same narrow two-day window, several submissions only minutes apart. This is consistent with a single short-lived event (a viral post, a Twitter Space, an influencer mention, or a bulk data entry event) rather than 29 individuals organically discovering the program independently.

**Implication for channel analysis:** X (Twitter)'s overall registration count is likely inflated by this one cluster. Without separating it out, X (Twitter) could appear to be a stronger organic channel than it actually is on a normal day. This will be treated as a distinct event in the channel analysis, not folded into "steady organic X (Twitter) conversion."

## Checking for Duplicate Registrants (Cohort 9)

The challenge brief confirms phone numbers were de-identified but consistently mapped: the same real person always produces the same replacement number. This makes Phone Number more reliable than Name or Email for catching duplicate registrations, especially since Name is missing on 55 rows.

Checking duplicates on Phone Number, Email, and the combination of both.

In [15]:
phone_col = 'Phone Number (Add country code e.g +234)'

print("Duplicate phone numbers:", cohort9[phone_col].duplicated().sum())
print("Duplicate emails:", cohort9['Email address'].duplicated().sum())
print("Duplicate on both phone AND email:", cohort9.duplicated(subset=[phone_col, 'Email address']).sum())

Duplicate phone numbers: 729
Duplicate emails: 191
Duplicate on both phone AND email: 43


## Investigating the Gap: 729 Duplicate Phones vs. 43 Duplicate Phone+Email

The sharp drop between phone-only duplicates (729) and phone+email duplicates (43) suggests most phone matches are coincidental collisions, not the same person registering twice, likely caused by short, de-identified phone numbers having a limited range of possible values.

Checking the length distribution of phone numbers to confirm.

In [16]:
phone_lengths = cohort9[phone_col].astype(str).str.replace(r'\D', '', regex=True).str.len()
phone_lengths.value_counts().sort_index()

Phone Number (Add country code e.g +234)
0      2
2      6
3     16
4    106
5    857
6    160
Name: count, dtype: int64

In [17]:
# Look at a few "duplicate phone" rows where email does NOT match — are these clearly different people?
mismatched_dups = cohort9[cohort9[phone_col].duplicated(keep=False)]
mismatched_dups = mismatched_dups[~mismatched_dups.duplicated(subset=[phone_col, 'Email address'], keep=False)]
mismatched_dups.sort_values(phone_col).head(15)

,Timestamp,NAME,Email address,Phone Number (Add country code e.g +234),"Country (e.g. Nigeria, UK)",Gender,Choice of Program,How did you hear about our Program?,Amount Paid
337,11/20/25 6:14 PM,Steven,stev@gmail.com,+14372,Canada,Male,Sales and Marketing Analytics,X (Twitter),NaN
937,1/23/26 1:03 AM,Tolulope,amos@gmail.com,+14372,Canada,Male,Financial Analytics,X (Twitter),NaN
302,11/17/25 8:52 PM,Isaac,arou@gmail.com,+21192,South Sudan,Male,Sales and Marketing Analytics,X (Twitter),NaN
475,12/15/25 2:52 PM,Nhial,mawi@gmail.com,+21192,South Sudan,Male,Data Science and AI,LinkedIn,NaN
770,1/22/26 8:19 PM,Alfred,b.fo@gmail.com,+233,Ghana,Male,Sales and Marketing Analytics,X (Twitter),NaN
530,12/29/25 3:04 PM,Benjamin,bens@gmail.com,+233,Ghana,Male,Financial Analytics,X (Twitter),NaN
432,12/2/25 2:43 PM,Francis,geny@gmail.com,+233 0,Ghana,Male,Healthcare Data Analytics,LinkedIn,NaN
733,1/22/26 7:43 PM,Jemimah,jemi@gmail.com,+233 0,Ghana,Female,Sales and Marketing Analytics,Referral,NaN
375,11/24/25 8:11 PM,Felix,Feli@gmail.com,+233 2,Ghana,Male,Data Science and AI,X (Twitter),NaN
929,1/23/26 12:30 AM,Lucas,luca@gmail.com,+233 2,Ghana,Male,Financial Analytics,X (Twitter),NaN


## Conclusion: Phone Number Alone Is Not a Reliable Duplicate Check

Phone numbers in this dataset are almost entirely 5-6 digits long (857 rows at 5 digits, 160 at 6 digits). This is a narrow enough range that unrelated people collide on the same number by chance.

Examples confirm this directly: rows sharing a phone number show different names, different countries, and registrations months apart (e.g. Steven vs. Tolulope on the same number; ten different Ghanaian registrants across different courses and dates sharing near-identical numbers).

**Decision:** Duplicate detection will use the combination of Phone Number AND Email Address (43 matching rows), not Phone Number alone (729 matching rows). Matching on both fields together makes an accidental collision far less likely, since it would require two unrelated people to coincidentally share both a short de-identified number and an email username.

In [18]:
true_dups = cohort9[cohort9.duplicated(subset=[phone_col, 'Email address'], keep=False)]
true_dups.sort_values([phone_col, 'Timestamp'])

,Timestamp,NAME,Email address,Phone Number (Add country code e.g +234),"Country (e.g. Nigeria, UK)",Gender,Choice of Program,How did you hear about our Program?,Amount Paid
179,11/12/25 6:26 PM,Mawemuko,joan@gmail.com,+1 587,Canada,Female,Healthcare Data Analytics,LinkedIn,NaN
180,11/12/25 6:27 PM,Mawemuko,joan@gmail.com,+1 587,Canada,Female,Data Science and AI,LinkedIn,NaN
1068,1/23/26 12:24 PM,Adeyanju Hammed,hamm@gmail.com,+234 7,Nigeria,Male,Data Science and AI,X (Twitter),NaN
405,11/25/25 8:17 PM,ADEYANJU,hamm@gmail.com,+234 7,Nigeria,Male,Sales and Marketing Analytics,X (Twitter),NaN
664,1/22/26 5:44 PM,Chika,Ugoc@gmail.com,+234 8,Nigeria,Male,Data Science and AI,X (Twitter),NaN
...,...,...,...,...,...,...,...,...,...
607,12/30/25 8:19 PM,Abdulkareem,abdu@gmail.com,081168,Nigeria,Female,Financial Analytics,LinkedIn,NaN
431,12/2/25 12:45 PM,Oluwafemi,aded@gmail.com,081624,Nigeria,Male,Healthcare Data Analytics,X (Twitter),NaN
528,12/29/25 2:55 PM,Adedokun,aded@gmail.com,081624,Nigeria,Male,Healthcare Data Analytics,X (Twitter),NaN
581,12/30/25 5:37 AM,Maroa,maro@gmail.com,254711,Kenya,Male,Financial Analytics,X (Twitter),NaN


## Duplicate Registrants: Not Noise, a Behavior Pattern

The 43 confirmed duplicate rows (matching on Phone + Email) are not accidental repeats. They show real people registering more than once, frequently changing their Choice of Program between submissions (sometimes minutes apart, sometimes months apart). In a few cases the name is entered differently between submissions (e.g. "ADEYANJU" vs "Adeyanju Hammed"), and in a couple of cases the name differs entirely on a shared phone/email, possibly indicating a shared household contact rather than data error.

**Decision:** For duplicate registrants, the most recent submission will be treated as their final registration (their latest course choice reflects their actual decision), and earlier submissions from the same person will be excluded from headcount to avoid inflating registration numbers. The fact that course-switching happens at all will be kept as a separate finding, since it may be relevant to the funnel/growth recommendation.

In [19]:
n_dup_groups = cohort9.duplicated(subset=[phone_col, 'Email address'], keep=False).sum()
n_unique_people_in_dups = cohort9.groupby([phone_col, 'Email address']).ngroups
total_dup_rows = cohort9.duplicated(subset=[phone_col, 'Email address'], keep='first').sum()

print("Rows involved in duplicate groups:", n_dup_groups)
print("Rows that are 'extra' (will be removed, keeping most recent):", total_dup_rows)

Rows involved in duplicate groups: 80
Rows that are 'extra' (will be removed, keeping most recent): 43


In [20]:
rows_to_drop_correct = cohort9.duplicated(subset=[phone_col, 'Email address'], keep='last').sum()
print("Rows that will actually be removed (keeping most recent):", rows_to_drop_correct)

n_groups = cohort9.groupby([phone_col, 'Email address']).ngroups
print("Number of distinct people with duplicate registrations:", cohort9[cohort9.duplicated(subset=[phone_col, 'Email address'], keep=False)].groupby([phone_col, 'Email address']).ngroups)

Rows that will actually be removed (keeping most recent): 43
Number of distinct people with duplicate registrations: 37


## Duplicate Removal: Confirmed Numbers

80 rows belong to 37 distinct people (matched on Phone + Email) who registered more than once. Removing the earlier submissions and keeping each person's most recent one drops 43 rows, leaving 37 rows, one per person. Since 37 people accounted for 80 rows rather than exactly 74 (37 x 2), a small number of people registered three or more times.

Final Cohort 9 row count after deduplication: 1147 - 43 = 1104 rows.

## Dropping Amount Paid, Converting Timestamp

In [21]:
cohort9 = cohort9.drop(columns=['Amount Paid'])
cohort9['Timestamp'] = pd.to_datetime(cohort9['Timestamp'], format='%m/%d/%y %I:%M %p')

cohort9[['Timestamp']].head()
cohort9['Timestamp'].dtype

dtype('<M8[ns]')

## Applying Deduplication

Removing earlier duplicate submissions, keeping each person's most recent registration (matched on Phone + Email).

In [22]:
before = len(cohort9)
cohort9 = cohort9.drop_duplicates(subset=[phone_col, 'Email address'], keep='last').reset_index(drop=True)
after = len(cohort9)

print(f"Rows before: {before}")
print(f"Rows after: {after}")
print(f"Rows removed: {before - after}")

Rows before: 1147
Rows after: 1104
Rows removed: 43


## Country Standardization (Cohort 9)

Checking all unique values in the Country field to identify trailing spaces, truncation, casing inconsistencies, and any free-text variations before deciding how to standardize.

In [23]:
country_col = 'Country (e.g. Nigeria, UK)'

unique_countries = cohort9[country_col].unique()
print("Number of unique raw values:", len(unique_countries))
sorted(unique_countries)

Number of unique raw values: 105


['+27 81 313 5587',
 '8 056 132997 ',
 'Belarus (Nigerian)',
 'Benin Republic ',
 'Botswana ',
 'Botswana Africa ',
 'Brazil ',
 'Cameroon',
 'Cameroon ',
 'Canada',
 'China',
 'Colombia ',
 'Cote D ivoire',
 'Democratic republic of the Congo',
 'Egypt ',
 'España',
 'Ethiopia',
 'Ethiopia ',
 'GHANA',
 'GHANA ',
 'Gambia',
 'Germany',
 'Ghan',
 'Ghana',
 'Ghana ',
 'INDIA ',
 'India',
 'India ',
 'Ireland ',
 'Ivory Coast',
 'Jamaica',
 'KENYA',
 'KENYA ',
 'Kenya',
 'Kenya ',
 'Kuwait',
 'Lesotho',
 'Liberia',
 'Malawi',
 'Malawi ',
 'Mozambique ',
 'NIGERIA ',
 'Namibia',
 'Namibia ',
 'Netherlands ',
 'Nig',
 'Nige',
 'Niger',
 'Nigera ',
 'Nigeria',
 'Nigeria ',
 'Nigeria Ekiti state ',
 'Nigeria Nigeria ',
 'Nigeria l',
 'Nigeria, Russia. ',
 'Nigerian ',
 'Nigerianl',
 'Peru',
 'RSA',
 'RWANDA ',
 'Rwanda',
 'SA',
 'Saudi Arabia ',
 'Senegal',
 'Senegal ',
 'Sierra Leone',
 'Sierra Leone ',
 'Somalia',
 'Somalia ',
 'Sout',
 'South Africa',
 'South Africa ',
 'South African ',
 

## Country Cleaning Plan

Raw values fall into distinct categories:
1. Whitespace/casing differences (e.g. "Ghana " vs "GHANA" vs "ghana")
2. Abbreviations (RSA, SA, UK, US, USA)
3. Alternate names for the same country (Cote D Ivoire / Ivory Coast)
4. Clear truncation (Nig, Nige, Ghan)
5. Non-country entries: phone numbers typed into the field, and the literal placeholder text "e.g." left in by a respondent
6. Genuinely ambiguous entries that cannot be safely resolved (e.g. "Niger" could be the country Niger or a truncated "Nigeria"; "Sout" could be South Africa or South Sudan; "U" is unresolvable)
7. Multiple values in one field (e.g. "Nigeria, Russia.")

Categories 1 to 4 will be standardized via a mapping. Category 5 will be set to a clear "Invalid Entry" label rather than dropped, so the row is not lost for other analysis. Category 6 will be labeled "Ambiguous" rather than guessed. Category 7 will be handled case by case.

In [24]:
cohort9['country_clean'] = cohort9[country_col].str.strip().str.lower()
print("Unique values after strip/lower:", cohort9['country_clean'].nunique())
sorted(cohort9['country_clean'].unique())

Unique values after strip/lower: 69


['+27 81 313 5587',
 '8 056 132997',
 'belarus (nigerian)',
 'benin republic',
 'botswana',
 'botswana africa',
 'brazil',
 'cameroon',
 'canada',
 'china',
 'colombia',
 'cote d ivoire',
 'democratic republic of the congo',
 'e.g.',
 'egypt',
 'españa',
 'ethiopia',
 'gambia',
 'germany',
 'ghan',
 'ghana',
 'india',
 'ireland',
 'ivory coast',
 'jamaica',
 'kenya',
 'kuwait',
 'lesotho',
 'liberia',
 'malawi',
 'mozambique',
 'namibia',
 'netherlands',
 'nig',
 'nige',
 'niger',
 'nigera',
 'nigeria',
 'nigeria ekiti state',
 'nigeria l',
 'nigeria nigeria',
 'nigeria, russia.',
 'nigerian',
 'nigerianl',
 'peru',
 'rsa',
 'rwanda',
 'sa',
 'saudi arabia',
 'senegal',
 'sierra leone',
 'somalia',
 'sout',
 'south africa',
 'south african',
 'south sudan',
 'sweden',
 'tanzania',
 'tanzania,dar es salaam',
 'turkey',
 'u',
 'uganda',
 'uk',
 'united kingdom',
 'united states',
 'us',
 'usa',
 'zambia',
 'zimbabwe']

## Country Mapping

Building an explicit mapping from every raw cleaned value to a standardized country name. Confident truncations/typos are mapped directly (e.g. "ghan" -> Ghana, "nig"/"nige"/"nigera" -> Nigeria). Non-country entries (phone numbers, leftover placeholder text) are labeled "Invalid Entry". Entries that cannot be safely resolved to one country are labeled "Ambiguous" rather than guessed.

In [25]:
country_map = {
    # Nigeria (confident truncations/typos)
    'nig': 'Nigeria', 'nige': 'Nigeria', 'nigera': 'Nigeria', 'nigeria': 'Nigeria',
    'nigeria ekiti state': 'Nigeria', 'nigeria l': 'Nigeria', 'nigeria nigeria': 'Nigeria',
    'nigerian': 'Nigeria', 'nigerianl': 'Nigeria',

    # Ghana
    'ghan': 'Ghana', 'ghana': 'Ghana',

    # South Africa
    'south africa': 'South Africa', 'south african': 'South Africa', 'rsa': 'South Africa',

    # UK / US
    'uk': 'United Kingdom', 'united kingdom': 'United Kingdom',
    'us': 'United States', 'usa': 'United States', 'united states': 'United States',

    # Ivory Coast
    'cote d ivoire': 'Ivory Coast', 'ivory coast': 'Ivory Coast',

    # Other direct matches (already clean after strip/lower)
    'benin republic': 'Benin',
    'botswana': 'Botswana', 'botswana africa': 'Botswana',
    'brazil': 'Brazil', 'cameroon': 'Cameroon', 'canada': 'Canada', 'china': 'China',
    'colombia': 'Colombia', 'democratic republic of the congo': 'DR Congo',
    'egypt': 'Egypt', 'españa': 'Spain', 'ethiopia': 'Ethiopia', 'gambia': 'Gambia',
    'germany': 'Germany', 'india': 'India', 'ireland': 'Ireland', 'jamaica': 'Jamaica',
    'kenya': 'Kenya', 'kuwait': 'Kuwait', 'lesotho': 'Lesotho', 'liberia': 'Liberia',
    'malawi': 'Malawi', 'mozambique': 'Mozambique', 'namibia': 'Namibia',
    'netherlands': 'Netherlands', 'peru': 'Peru', 'rwanda': 'Rwanda',
    'saudi arabia': 'Saudi Arabia', 'senegal': 'Senegal', 'sierra leone': 'Sierra Leone',
    'somalia': 'Somalia', 'south sudan': 'South Sudan', 'sweden': 'Sweden',
    'tanzania': 'Tanzania', 'tanzania,dar es salaam': 'Tanzania', 'turkey': 'Turkey',
    'uganda': 'Uganda', 'zambia': 'Zambia', 'zimbabwe': 'Zimbabwe',

    # Invalid entries (not countries at all)
    '+27 81 313 5587': 'Invalid Entry', '8 056 132997': 'Invalid Entry', 'e.g.': 'Invalid Entry',

    # Ambiguous - cannot safely resolve
    'niger': 'Ambiguous', 'sa': 'Ambiguous', 'sout': 'Ambiguous', 'u': 'Ambiguous',
    'nigeria, russia.': 'Ambiguous', 'belarus (nigerian)': 'Ambiguous',
}

cohort9['country_clean'] = cohort9['country_clean'].map(country_map)
cohort9['country_clean'].value_counts(dropna=False)

country_clean
Nigeria           621
South Africa      106
Ghana             106
Kenya              91
Uganda             30
United Kingdom     20
Zimbabwe           19
India              13
United States      12
Canada             10
Ambiguous           7
Malawi              7
Tanzania            6
Zambia              6
Cameroon            5
Rwanda              4
Ethiopia            4
Invalid Entry       3
Botswana            2
Senegal             2
South Sudan         2
Namibia             2
Ivory Coast         2
Sierra Leone        2
Somalia             2
Benin               1
Gambia              1
China               1
Jamaica             1
Spain               1
Mozambique          1
Saudi Arabia        1
DR Congo            1
Egypt               1
Colombia            1
Netherlands         1
Peru                1
Germany             1
Liberia             1
Kuwait              1
Turkey              1
Brazil              1
Lesotho             1
Sweden              1
Ireland           

In [26]:
print("Rows with unmapped (NaN) country_clean:", cohort9['country_clean'].isna().sum())
cohort9['country_clean'].value_counts(dropna=False).sum(), len(cohort9)

Rows with unmapped (NaN) country_clean: 0


(np.int64(1104), 1104)

## Country Cleaning: Result

Standardized Country field down to a clean set of country names. Nigeria accounts for the majority of registrations (621 of 1104, ~56%), with South Africa, Ghana, and Kenya as the next largest groups. 7 rows (0.6%) are labeled "Ambiguous" (values that could not be safely resolved to one country) and 3 rows (0.3%) are labeled "Invalid Entry" (phone numbers or placeholder text typed into the Country field). These 10 rows are kept in the dataset but excluded from any country-specific breakdown, since including them would misrepresent the data.

# Cohort 10: Cleaning

Cohort 10 is a smaller, cleaner dataset (171 rows), but we run the same checks rather than assuming it's fine: missing values, duplicates, timestamp conversion, and country standardization.

## Checking the Two Missing Values

`.info()` showed 1 missing Phone Number and 1 missing Occupation. Looking at those specific rows before deciding how to handle them.

In [27]:
phone10 = 'PHONE NUMBER  (Add country code e.g +234)'
country10 = 'COUNTRY  (e.g. Nigeria, UK)'

cohort10[cohort10[phone10].isna()]

,Timestamp,NAME,Gender,EMAIL ADDRESS,PHONE NUMBER (Add country code e.g +234),"COUNTRY (e.g. Nigeria, UK)",CHOICE OF COURSE,Occupation,How did you hear about our Program?
14,5/14/2026 16:26:33,Julius,Male,juli@gmail.com,NaN,Ghana,Financial Analytics,Customer Service,X (Twitter)


In [30]:
cohort10[cohort10['Occupation'].isna()]

,Timestamp,NAME,Gender,EMAIL ADDRESS,PHONE NUMBER (Add country code e.g +234),"COUNTRY (e.g. Nigeria, UK)",CHOICE OF COURSE,Occupation,How did you hear about our Program?
162,5/29/2026 1:58:39,Genesis,Female,Gene@gmail.com,951419,United states,AI Automation,NaN,X (Twitter)


## Missing Values: Result

Unlike Cohort 9, these two missing values are isolated and unrelated to each other. One row (Julius, Ghana) is missing Phone Number only; another row (Genesis, United States) is missing Occupation only. No pattern connects them, consistent with two respondents each skipping a single form field.

**Decision:** Leave both as missing (NaN) rather than guessing a value. Phone Number is not needed for this analysis. Occupation missing on one row out of 171 has negligible impact on any occupation-based breakdown.

## Duplicates and Timestamp (Cohort 10)

In [33]:
email10 = 'EMAIL ADDRESS'
print("Duplicate on Phone + Email:", cohort10.duplicated(subset=[phone10, email10]).sum())

cohort10['Timestamp'] = pd.to_datetime(cohort10['Timestamp'], format='%m/%d/%Y %H:%M:%S')
cohort10['Timestamp'].dtype

Duplicate on Phone + Email: 3


dtype('<M8[ns]')

## Timestamp and Duplicates: Result

Cohort 10 uses a different timestamp format than Cohort 9 (full 4-digit year, 24-hour clock, e.g. "5/6/2026 11:03:32" vs Cohort 9's "11/10/25 12:16 PM"). Both are now converted to proper datetime values. This format inconsistency between cohorts is documented as an assumption to watch for if the two datasets are ever combined by timestamp.

3 duplicate registrations found (matched on Phone + Email), consistent with the same registration-and-reconsider behavior seen in Cohort 9, just at a much smaller scale given Cohort 10 is 1/6th the size.

In [34]:
cohort10[cohort10.duplicated(subset=[phone10, email10], keep=False)].sort_values([phone10, 'Timestamp'])

,Timestamp,NAME,Gender,EMAIL ADDRESS,PHONE NUMBER (Add country code e.g +234),"COUNTRY (e.g. Nigeria, UK)",CHOICE OF COURSE,Occupation,How did you hear about our Program?
126,2026-05-15 12:08:52,Jorge,Male,jorg@gmail.com,+23480,Nig,Data science and ML,Freelance writer,X (Twitter)
156,2026-05-28 19:43:59,Jorge,Male,jorg@gmail.com,+23480,Nigeria,Sales and Marketing Analytics,Freelancer,X (Twitter)
8,2026-05-08 22:23:00,Damiete,Female,dami@gmail.com,+23481,Nigeria,Sales and Marketing Analytics,Post graduate,WhatsApp Community
131,2026-05-15 16:07:37,Joshua,Male,dami@gmail.com,+23481,Nigeria,Data science and ML,Student,X (Twitter)
145,2026-05-17 16:14:10,Akintola,Female,akin@gmail.com,+23481,Nigeria,Financial Analytics,Finance Analyst,Linkedin
154,2026-05-28 17:54:05,Akinloye,Male,akin@gmail.com,+23481,+2348133105464,Data science and ML,student,X (Twitter)


In [35]:
before10 = len(cohort10)
cohort10 = cohort10.drop_duplicates(subset=[phone10, email10], keep='last').reset_index(drop=True)
after10 = len(cohort10)

print(f"Rows before: {before10}")
print(f"Rows after: {after10}")
print(f"Rows removed: {before10 - after10}")

Rows before: 171
Rows after: 168
Rows removed: 3


## Duplicates: Result

3 duplicate registrations removed (171 -> 168), keeping the most recent submission per person. Same course-switching pattern observed as in Cohort 9. One duplicate row's Country field also contained a phone number typed in by mistake, the same error type seen in Cohort 9, confirming Country needs the same cleanup here.

In [36]:
cohort10['country_clean'] = cohort10[country10].str.strip().str.lower()
print("Unique values:", cohort10['country_clean'].nunique())
sorted(cohort10['country_clean'].unique())

Unique values: 25


['+2348133105464',
 'cameroon',
 'canada',
 'côte d’ivoire',
 'ethiopia',
 'ghana',
 'india',
 'ireland',
 'kebya',
 'kenya',
 'nam',
 'namibia',
 'ng',
 'nigeria',
 'nigeria l',
 'nigerian',
 'sierra leone',
 'south africa',
 'south aftrica',
 'uganda',
 'uk',
 'united kingdom',
 'united states',
 'usa',
 'zimbabwe']

In [37]:
country_map_10 = {
    'nigeria': 'Nigeria', 'nigeria l': 'Nigeria', 'nigerian': 'Nigeria', 'ng': 'Nigeria',
    'ghana': 'Ghana',
    'south africa': 'South Africa', 'south aftrica': 'South Africa',
    'uk': 'United Kingdom', 'united kingdom': 'United Kingdom',
    'usa': 'United States', 'united states': 'United States',
    'côte d\u2019ivoire': 'Ivory Coast',
    'kenya': 'Kenya', 'kebya': 'Kenya',
    'namibia': 'Namibia', 'nam': 'Namibia',
    'cameroon': 'Cameroon', 'canada': 'Canada', 'ethiopia': 'Ethiopia',
    'india': 'India', 'ireland': 'Ireland', 'sierra leone': 'Sierra Leone',
    'uganda': 'Uganda', 'zimbabwe': 'Zimbabwe',

    '+2348133105464': 'Invalid Entry',
}

cohort10['country_clean'] = cohort10['country_clean'].map(country_map_10)
print("Unmapped rows:", cohort10['country_clean'].isna().sum())
cohort10['country_clean'].value_counts(dropna=False)

Unmapped rows: 0


country_clean
Nigeria           103
Kenya              20
South Africa       11
Ghana               9
Uganda              6
Namibia             4
United States       3
Zimbabwe            2
United Kingdom      2
Ivory Coast         1
Ireland             1
Canada              1
Sierra Leone        1
India               1
Cameroon            1
Ethiopia            1
Invalid Entry       1
Name: count, dtype: int64

## Cohort 10 Country Cleaning: Result

Standardized Country field with zero unmapped rows. Nigeria accounts for 103 of 168 registrations (~61%), a higher concentration than Cohort 9 (~56%). Only 1 row (0.6%) is "Invalid Entry" (a phone number typed into the Country field). No "Ambiguous" entries this time, all typos were confidently resolvable.

# Cleaning Complete

Both datasets are cleaned and ready for exploratory analysis. Exported to `data/processed/` for use in the next notebook.

In [38]:
cohort9.to_csv('../data/processed/cohort9_clean.csv', index=False)
cohort10.to_csv('../data/processed/cohort10_clean.csv', index=False)

print("Saved cohort9_clean.csv:", cohort9.shape)
print("Saved cohort10_clean.csv:", cohort10.shape)

Saved cohort9_clean.csv: (1104, 9)
Saved cohort10_clean.csv: (168, 10)
